# Experiment 01C : Modular Collagen Architecture

**Goal** : To create a simplified tumor ECM architecture with spatial variation while keeping every biological assumption adjustable.

This will consist of three main tissue components : 

1. Tumor-dominant tissue (Tumor-cell-rich regions), Lower relative collagen
2. Stromal regions (ECM-rich areas), Elevated relative collagen
3. Local collagen structures (Fiborous bands/aggregates), spatially localized collagen enrichment

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# EXPERIMENT 01C — COLLAGEN ARCHITECTURE GENERATION
# ============================================================

# ----------------------------
# 1. Domain and grid
# ----------------------------

Lx_um = 100.0   # distance from vessel (um)
Ly_um = 200.0   # parallel distance (um)

# Match the grid resolution used in the diffusion simulation
Nx, Ny = 200, 100

x_um = np.linspace(0, Lx_um, Nx)
y_um = np.linspace(0, Ly_um, Ny)

X, Y = np.meshgrid(x_um, y_um, indexing="ij")

# ----------------------------
# 2. Collagen parameters
# ----------------------------

rho_mean_target = 0.60
rho_min = 0.0
rho_max = 1.0

# ----------------------------
# 3. Mean-matching function
# ----------------------------

def match_mean_bounded(field, target_mean, lower=0.0, upper=1.0):

    field = np.asarray(field, dtype=float)

    lo = -2.0
    hi = 2.0

    for _ in range(200):

        shift = (lo + hi) / 2
        candidate = np.clip(field + shift, lower, upper)

        if candidate.mean() < target_mean:
            lo = shift
        else:
            hi = shift

    shift = (lo + hi) / 2

    matched = np.clip(field + shift, lower, upper)

    return matched


# ============================================================
# 4. ARCHITECTURE A — UNIFORM COLLAGEN
# ============================================================

rho_uniform = np.full(
    (Nx, Ny),
    rho_mean_target,
    dtype=float
)


# ============================================================
# 5. ARCHITECTURE B — PATCHY COLLAGEN
# ============================================================

patchy_raw = (
    0.80 * np.exp(
        -(
            (X - 25)**2 / (2 * 14**2))
            +
            (Y - 70)**2 / (2 * 30**2)
        )
    +
    0.60 * np.exp(
        -(
            (X - 75)**2 / (2 * 20**2))
            +
            (Y - 145)**2 / (2 * 25**2)
        )
    +
    0.35 * np.exp(
        -(
            (X - 40)**2 / (2 * 25**2))
            +
            (Y - 120)**2 / (2 * 18**2)
        )
    )

rho_patchy = match_mean_bounded(
    patchy_raw,
    rho_mean_target,
    rho_min,
    rho_max
)


# ============================================================
# 6. ARCHITECTURE C — COLLAGEN BAND
# ============================================================

band_raw = (
    np.exp(
        -((X - 55)**2 / (2 * 12**2)))
    *
    (0.85 + 0.15 * np.cos(2 * np.pi * Y / Ly_um))
)

rho_band = match_mean_bounded(
    band_raw,
    rho_mean_target,
    rho_min,
    rho_max
)


# ============================================================
# 7. COLLECT ARCHITECTURES
# ============================================================

collagen_fields = {
    "Uniform": rho_uniform,
    "Patchy": rho_patchy,
    "Band": rho_band
}


# ============================================================
# 8. VISUALIZE COLLAGEN ARCHITECTURES
# ============================================================

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 5),
    constrained_layout=True
)

for ax, (name, field) in zip(
    axes,
    collagen_fields.items()
):

    im = ax.imshow(
        field.T,
        origin="lower",
        extent=[0, Lx_um, 0, Ly_um],
        aspect="auto",
        cmap="viridis",
        vmin=0,
        vmax=1
    )

    ax.set_title(f"{name} Collagen Architecture")
    ax.set_xlabel("Distance from vessel (µm)")
    ax.set_ylabel("Parallel distance (µm)")

fig.colorbar(
    im,
    ax=axes,
    label="Normalized collagen density",
    shrink=0.85
)

plt.show()


# ============================================================
# 9. ARCHITECTURE STATISTICS
# ============================================================

print("=" * 70)
print("COLLAGEN ARCHITECTURE SUMMARY")
print("=" * 70)

for name, field in collagen_fields.items():

    mean = field.mean()
    sd = field.std()
    cv = sd / mean if mean != 0 else np.nan

    print(f"\n{name}")
    print("-" * 40)
    print(f"Mean:  {mean:.6f}")
    print(f"SD:    {sd:.6f}")
    print(f"CV:    {cv:.6f}")
    print(f"Min:   {field.min():.6f}")
    print(f"Max:   {field.max():.6f}")
    print(f"Shape: {field.shape}")


# ============================================================
# 10. VALIDATION
# ============================================================

print("Target mean:", rho_mean_target)

for name, field in collagen_fields.items():
    print(
        f"{name:10s} | mean={field.mean():.10f} "
        f"| difference={field.mean() - rho_mean_target:+.10e} "
        f"| min={field.min():.6f} | max={field.max():.6f}"
    )

assert rho_uniform.shape == (200, 100)
assert rho_patchy.shape == (200, 100)
assert rho_band.shape == (200, 100)

assert all(
    field.min() >= rho_min and field.max() <= rho_max
    for field in collagen_fields.values()
)

assert all(
    abs(field.mean() - rho_mean_target) < 1e-6
    for field in collagen_fields.values()
)

print("\nVALIDATION PASSED")
print("All fields have the correct shape and bounded density.")
print("All collagen architectures have matched mean density.")
